# UWB Average Distance

Calculate the average UWB distance from manually entered values or from logger CSV files.

In [ ]:
from pathlib import Path
import csv
import math


def clean_distances(values):
    distances = []

    for value in values:
        try:
            distance = float(value)
        except (TypeError, ValueError):
            continue

        if math.isfinite(distance):
            distances.append(distance)

    return distances


def summarize_distances(values):
    distances = clean_distances(values)

    if not distances:
        raise ValueError("No valid distance values were provided.")

    return {
        "count": len(distances),
        "average_distance_m": sum(distances) / len(distances),
        "min_distance_m": min(distances),
        "max_distance_m": max(distances),
    }


def read_distances_from_csv(csv_path, column="Dist", anchor=None):
    path = Path(csv_path)
    distances = []

    with path.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)

        if column not in (reader.fieldnames or []):
            raise ValueError(f"Column '{column}' was not found in {path}.")

        for row in reader:
            if anchor is not None and row.get("ANC") != str(anchor):
                continue
            distances.append(row.get(column))

    return distances


def summarize_by_anchor(csv_path, column="Dist"):
    path = Path(csv_path)
    grouped = {}

    with path.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)

        if "ANC" not in (reader.fieldnames or []):
            raise ValueError(f"Column 'ANC' was not found in {path}.")
        if column not in (reader.fieldnames or []):
            raise ValueError(f"Column '{column}' was not found in {path}.")

        for row in reader:
            grouped.setdefault(row.get("ANC"), []).append(row.get(column))

    return {
        anchor: summarize_distances(values)
        for anchor, values in sorted(grouped.items(), key=lambda item: item[0] or "")
        if clean_distances(values)
    }

## 1. Use Direct Distance Values

Replace the sample values with measured UWB distances in meters.

In [ ]:
distances = [1.20, 1.25, 1.30]

summarize_distances(distances)

## 2. Use Logger CSV

Put the UWB logger CSV file in `D:/uwb-config/data`, then write that file name in `CSV_FILE` below.

Example file name: `1_uwb_20260720_183000_123_KST.csv`

In [ ]:
repo_root = Path.cwd()
if repo_root.name.lower() == "calibration":
    repo_root = repo_root.parent

data_dir = repo_root / "data"

# Put the logger CSV file in D:/uwb-config/data and write its file name here.
CSV_FILE = "1_uwb_YYYYMMDD_HHMMSS_mmm_KST.csv"

csv_path = data_dir / CSV_FILE
csv_path

In [ ]:
anchor_id = None  # Example: set to 2 to calculate only anchor 2.

if not csv_path.exists():
    print(f"CSV file not found: {csv_path}")
    print("Change CSV_FILE to the actual file name in D:/uwb-config/data.")
else:
    distances_from_csv = read_distances_from_csv(csv_path, anchor=anchor_id)
    summarize_distances(distances_from_csv)

## 3. Average Distance By Anchor

Use this when the CSV contains measurements from multiple anchors.

In [ ]:
if not csv_path.exists():
    print(f"CSV file not found: {csv_path}")
    print("Change CSV_FILE to the actual file name in D:/uwb-config/data.")
else:
    summarize_by_anchor(csv_path)